# Phase 1 · Tokenizing the training corpus

**In simple words:** this notebook turns the four corpus sources into token ids **once** and stores them in compact binary files.
The model notebooks then read ids directly, instead of tokenizing text again in every training run.

**How it works:** every document is normalized, tokenized with the tokenizer from `01_bpe_tokenizer`, and gets an `<eos>` token
at its end (so the model learns where a document stops — the same `<eos>` later separates summaries in Phase 6). Ids are written
as `uint16` (2 bytes each, enough for 32,004 ids) to one file per source, `<source>_100m.bin` (up to 100M words per source),
plus a metadata file the model notebooks check.

**To run it:** attach the output of `01_bpe_tokenizer` as an input and add the Kaggle secret `HF`.

### CELL 1 · Imports + configuration

**In simple words:** Imports, paths (the tokenizer input, the output folder) and the budget of up to 100M words per source.

In [1]:
# ============================================================
# CELL 1 — IMPORTS + CONFIGURATION
# ============================================================
!pip install ijson
import sys
import json
import time
from pathlib import Path
from collections import Counter

import numpy as np
import ijson


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer"
)

OUTPUT_DIR = Path(
    "/kaggle/working/legal_tokenized_100m"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Tokenization configuration
# ------------------------------------------------------------

MAX_WORDS_PER_SOURCE = 100_000_000

EOS_TOKEN = "<eos>"
BLOCK_SIZE = 2048

DTYPE = np.uint16


# ------------------------------------------------------------
# Add saved tokenizer to Python path
# ------------------------------------------------------------

sys.path.insert(
    0,
    str(TOKENIZER_DIR),
)

from tokenizer import load_tokenizer


print("Tokenizer directory:", TOKENIZER_DIR)
print("Output directory   :", OUTPUT_DIR)
print("Target words/source:", f"{MAX_WORDS_PER_SOURCE:,}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 2.0 MB/s eta 0:00:00
Tokenizer directory: /kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/legal_tokenizer_v3/tokenizer/final_tokenizer
Output directory   : /kaggle/working/legal_tokenized_100m
Target words/source: 100,000,000


### CELL 2 · Load the tokenizer

**In simple words:** Load the tokenizer trained in `01_bpe_tokenizer`.

In [2]:
# ============================================================
# CELL 2 — LOAD FINAL BPE TOKENIZER
# ============================================================

tokenizer, tokenizer_config = load_tokenizer(
    TOKENIZER_DIR
)

print("=" * 70)
print("TOKENIZER LOADED")
print("=" * 70)

print(
    "Vocabulary size:",
    tokenizer.get_vocab_size(),
)

print(
    "Number of merges:",
    len(tokenizer.merges),
)

print(
    "UNK ID:",
    tokenizer.unk_id,
)

print(
    "BOS ID:",
    tokenizer.bos_id,
)

print(
    "EOS ID:",
    tokenizer.eos_id,
)

assert tokenizer.get_vocab_size() == 32004
assert tokenizer.eos_id is not None

print("\n✓ Tokenizer ready")

TOKENIZER LOADED
Vocabulary size: 32004
Number of merges: 31438
UNK ID: 1
BOS ID: 2
EOS ID: 3

✓ Tokenizer ready


### Hugging Face login

**In simple words:** Log in to Hugging Face with the token stored as the Kaggle secret `HF` (the dataset is gated).

**How it works:** The token is read with `UserSecretsClient().get_secret("HF")` and is never written in the notebook.

In [3]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret("HF")

os.environ["HF_TOKEN"] = HF_TOKEN

login(token=HF_TOKEN)

print("Hugging Face login successful.")

Hugging Face login successful.


### CELL 3 · Download the training files

**In simple words:** Download the four training files.

In [4]:
# ============================================================
# CELL 3 — DOWNLOAD TRAINING FILES
# ============================================================

from huggingface_hub import hf_hub_download


HF_DATASET_REPO = (
    "Exploration-Lab/CS-781-Capstone"
)

TRAIN_FILES = {
    "acts":
        "Phase_1_Pretraining/Train/train_acts.json",

    "high_courts":
        "Phase_1_Pretraining/Train/train_high_courts.json",

    "supreme_court":
        "Phase_1_Pretraining/Train/train_supreme_court.json",

    "tribunals":
        "Phase_1_Pretraining/Train/train_tribunals.json",
}


downloaded_files = {}

for source_name, filename in TRAIN_FILES.items():

    print(
        f"\nDownloading/locating {source_name}..."
    )

    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
    )

    downloaded_files[source_name] = path

    print(
        "Path:",
        path,
    )


print("\n✓ All training files ready")


Downloading/locating acts...


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json

Downloading/locating high_courts...


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_high_courts.json

Downloading/locating supreme_court...


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json

Downloading/locating tribunals...


Path: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

✓ All training files ready


### CELL 4 · Normalization

**In simple words:** The same text cleaning as in the tokenizer notebook.

In [5]:
# ============================================================
# CELL 4 — NORMALIZATION
# ============================================================

import unicodedata


def normalize_text(text: str) -> str:

    if not isinstance(text, str):
        text = str(text)

    text = unicodedata.normalize(
        "NFKC",
        text,
    )

    text = text.replace(
        "\r\n",
        "\n",
    )

    text = text.replace(
        "\r",
        "\n",
    )

    text = text.replace(
        "\x00",
        "",
    )

    return text

### CELL 5 · Word-like pieces

**In simple words:** Which pieces count as words — used to stop at the word budget of a source.

In [6]:
# ============================================================
# CELL 5 — WORD-LIKE PIECE DEFINITION
# ============================================================

def is_word_like_piece(piece):

    return bool(
        piece.strip()
        and any(
            ch.isalnum()
            for ch in piece
        )
    )

### CELL 6 · Stream JSON documents

**In simple words:** Read one document at a time from the huge JSON files.

In [7]:
# ============================================================
# CELL 6 — STREAM JSON DOCUMENTS
# ============================================================

def iter_json_documents(path):

    with open(
        path,
        "rb",
    ) as f:

        for record in ijson.items(
            f,
            "item",
        ):

            if not isinstance(
                record,
                dict,
            ):
                continue

            text = record.get(
                "text"
            )

            if (
                isinstance(text, str)
                and text
            ):

                yield text

### CELL 7 · Tokenize one document

**In simple words:** Normalize and tokenize one document and add `<eos>` at its end, so the model knows where a document stops.

In [8]:
# ============================================================
# CELL 7 — TOKENIZE ONE DOCUMENT
# ============================================================

def tokenize_document(
    text,
    tokenizer,
):

    text = normalize_text(
        text
    )

    ids = tokenizer.encode(
        text
    )

    # Add EOS after EVERY document
    ids.append(
        tokenizer.eos_id
    )

    return ids

### CELL 8 · Streaming token writer

**In simple words:** Tokenize a whole source document by document and append the ids to its binary file.

**How it works:** `uint16` = 2 bytes per token; nothing has to fit in memory at once.

In [9]:
# ============================================================
# CELL 8 — STREAMING TOKEN WRITER
# ============================================================

def tokenize_source_to_bin(
    source_name,
    input_path,
    output_path,
    tokenizer,
    max_word_pieces,
    buffer_tokens=1_000_000,
):

    print("\n" + "=" * 70)
    print(
        f"TOKENIZING: {source_name}"
    )
    print("=" * 70)

    start_time = time.time()

    documents = 0
    word_pieces = 0
    token_count = 0
    unknown_count = 0

    buffer = []

    # --------------------------------------------------------
    # Open output file
    # --------------------------------------------------------

    with open(
        output_path,
        "wb",
    ) as out:

        # ----------------------------------------------------
        # Stream documents
        # ----------------------------------------------------

        for text in iter_json_documents(
            input_path
        ):

            normalized = normalize_text(
                text
            )

            # ------------------------------------------------
            # Count word-like pieces
            # ------------------------------------------------

            piece_count = 0

            for piece in tokenizer_module_pretokenize(
                normalized
            ):

                if is_word_like_piece(
                    piece
                ):
                    piece_count += 1

            # If adding this document would exceed
            # the requested budget, stop BEFORE the document.
            if (
                word_pieces + piece_count
                > max_word_pieces
            ):
                break

            # ------------------------------------------------
            # Encode document
            # ------------------------------------------------

            ids = tokenizer.encode(
                normalized
            )

            # EOS at document boundary
            ids.append(
                tokenizer.eos_id
            )

            # ------------------------------------------------
            # Validate token IDs
            # ------------------------------------------------

            if ids:

                local_max = max(
                    ids
                )

                if local_max >= tokenizer.get_vocab_size():

                    raise ValueError(
                        f"Invalid token ID "
                        f"{local_max} in "
                        f"{source_name}"
                    )

            local_unk = sum(
                1
                for token_id in ids
                if token_id == tokenizer.unk_id
            )

            unknown_count += local_unk

            # ------------------------------------------------
            # Add to buffer
            # ------------------------------------------------

            buffer.extend(
                ids
            )

            documents += 1
            word_pieces += piece_count
            token_count += len(ids)

            # ------------------------------------------------
            # Flush
            # ------------------------------------------------

            if len(buffer) >= buffer_tokens:

                np.asarray(
                    buffer,
                    dtype=np.uint16,
                ).tofile(
                    out
                )

                buffer.clear()

            # ------------------------------------------------
            # Progress
            # ------------------------------------------------

            if documents % 1000 == 0:

                elapsed = (
                    time.time()
                    - start_time
                )

                words_per_sec = (
                    word_pieces / elapsed
                    if elapsed > 0
                    else 0
                )

                print(
                    f"\r"
                    f"documents={documents:,} | "
                    f"words={word_pieces:,} / "
                    f"{max_word_pieces:,} | "
                    f"tokens={token_count:,} | "
                    f"{words_per_sec:,.0f} words/s",
                    end="",
                )

        # ----------------------------------------------------
        # Final flush
        # ----------------------------------------------------

        if buffer:

            np.asarray(
                buffer,
                dtype=np.uint16,
            ).tofile(
                out
            )

            buffer.clear()

    elapsed = (
        time.time()
        - start_time
    )

    file_size_gb = (
        output_path.stat().st_size
        / (1024 ** 3)
    )

    print("\n")

    print(
        "Documents      :",
        f"{documents:,}",
    )

    print(
        "Word-like pieces:",
        f"{word_pieces:,}",
    )

    print(
        "BPE tokens     :",
        f"{token_count:,}",
    )

    print(
        "UNK tokens     :",
        f"{unknown_count:,}",
    )

    print(
        "UNK rate       :",
        f"{unknown_count / token_count:.8%}"
        if token_count
        else "0",
    )

    print(
        "File size      :",
        f"{file_size_gb:.2f} GB",
    )

    print(
        "Time           :",
        f"{elapsed / 60:.2f} min",
    )

    return {
        "source": source_name,
        "documents": documents,
        "word_like_pieces": word_pieces,
        "tokens": token_count,
        "unk_tokens": unknown_count,
        "unk_rate": (
            unknown_count / token_count
            if token_count
            else 0.0
        ),
        "file_size_bytes":
            output_path.stat().st_size,
        "output_file":
            str(output_path),
        "elapsed_seconds":
            elapsed,
    }

### CELL 9 · Pre-tokenizer access

**In simple words:** Use the saved tokenizer's own pre-tokenizer (so counting words matches the tokenizer).

In [10]:
# ============================================================
# CELL 9 — TOKENIZER PRETOKENIZER ACCESS
# ============================================================

tokenizer_module_pretokenize = (
    tokenizer.__class__.__module__
)

# The saved tokenizer module exposes pretokenize().
import tokenizer as tokenizer_module

tokenizer_module_pretokenize = (
    tokenizer_module.pretokenize
)

print(
    "Pretokenizer loaded:"
)
print(
    tokenizer_module_pretokenize
)

Pretokenizer loaded:
<function pretokenize at 0x7b4b1dadd580>


### CELL 10 · Small sanity test

**In simple words:** Tokenize a short sample and check it.

**How it works:** Output removed before publishing.

In [11]:
# ============================================================
# CELL 10 — SMALL SANITY TEST
# ============================================================

sample_text = """
The Supreme Court held that the appeal
was dismissed.

Section 12 of the Act shall apply.
"""

sample_pieces = tokenizer_module_pretokenize(
    sample_text
)

sample_word_count = sum(
    1
    for piece in sample_pieces
    if is_word_like_piece(piece)
)

sample_ids = tokenizer.encode(
    sample_text
)

sample_ids.append(
    tokenizer.eos_id
)

print("Sample word-like pieces:", sample_word_count)
print("Sample token count      :", len(sample_ids))
print("EOS ID                  :", tokenizer.eos_id)
print("Last token              :", sample_ids[-1])

assert sample_ids[-1] == tokenizer.eos_id
assert all(
    0 <= x < tokenizer.get_vocab_size()
    for x in sample_ids
)

print("\n✓ Tokenization sanity test passed")

[output removed before publishing: this cell printed raw dataset / test text]


### CELL 11 · Tokenize all four sources

**In simple words:** Tokenize the four sources into `<source>_100m.bin`.

In [12]:
# ============================================================
# CELL 11 — TOKENIZE ALL FOUR SOURCES
# ============================================================

tokenization_stats = {}

for source_name, input_path in downloaded_files.items():

    output_path = (
        OUTPUT_DIR
        / f"{source_name}_100m.bin"
    )

    stats = tokenize_source_to_bin(
        source_name=source_name,
        input_path=input_path,
        output_path=output_path,
        tokenizer=tokenizer,
        max_word_pieces=MAX_WORDS_PER_SOURCE,
        buffer_tokens=1_000_000,
    )

    tokenization_stats[
        source_name
    ] = stats


TOKENIZING: acts
documents=15,000 | words=96,247,492 / 100,000,000 | tokens=247,310,740 | 128,807 words/s

Documents      : 15,490
Word-like pieces: 99,994,014
BPE tokens     : 256,521,619
UNK tokens     : 0
UNK rate       : 0.00000000%
File size      : 0.48 GB
Time           : 12.95 min

TOKENIZING: high_courts
documents=223,000 | words=99,765,602 / 100,000,000 | tokens=241,785,851 | 113,005 words/s

Documents      : 223,628
Word-like pieces: 99,999,403
BPE tokens     : 242,356,017
UNK tokens     : 0
UNK rate       : 0.00000000%
File size      : 0.45 GB
Time           : 14.74 min

TOKENIZING: supreme_court
documents=26,000 | words=97,090,600 / 100,000,000 | tokens=234,482,903 | 124,828 words/s

Documents      : 26,627
Word-like pieces: 99,978,684
BPE tokens     : 241,501,699
UNK tokens     : 0
UNK rate       : 0.00000000%
File size      : 0.45 GB
Time           : 13.37 min

TOKENIZING: tribunals
documents=57,000 | words=99,163,759 / 100,000,000 | tokens=239,683,759 | 115,158 words/s


### CELL 12 · Save metadata

**In simple words:** Save token counts, the tokenizer type and the special ids to `tokenization_metadata.json` — the model notebooks check these before training.

In [13]:
# ============================================================
# CELL 12 — SAVE TOKENIZATION METADATA
# ============================================================

metadata = {
    "tokenizer": {
        "type": "custom_bpe",
        "vocab_size":
            tokenizer.get_vocab_size(),
        "eos_token":
            EOS_TOKEN,
        "eos_id":
            tokenizer.eos_id,
        "dtype":
            "uint16",
    },

    "dataset": {
        "target_word_like_pieces_per_source":
            MAX_WORDS_PER_SOURCE,
        "block_size":
            BLOCK_SIZE,
    },

    "sources":
        tokenization_stats,
}


metadata_path = (
    OUTPUT_DIR
    / "tokenization_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
    )


print(
    "Metadata saved to:",
    metadata_path
)

Metadata saved to: /kaggle/working/legal_tokenized_100m/tokenization_metadata.json


### CELL 13 · Verify the tokenized data

**In simple words:** Open the files memory-mapped and check token counts, the id range and the number of `<eos>`.

In [14]:
# ============================================================
# CELL 13 — VERIFY TOKENIZED DATA
# ============================================================

print("=" * 70)
print("TOKENIZED DATA VERIFICATION")
print("=" * 70)

for source_name in downloaded_files:

    path = (
        OUTPUT_DIR
        / f"{source_name}_100m.bin"
    )

    assert path.exists(), (
        f"Missing file: {path}"
    )

    # Number of uint16 tokens
    token_count = (
        path.stat().st_size
        // np.dtype(
            np.uint16
        ).itemsize
    )

    # Memory-map without loading the entire file
    data = np.memmap(
        path,
        dtype=np.uint16,
        mode="r",
    )

    min_id = int(
        data.min()
    )

    max_id = int(
        data.max()
    )

    eos_count = int(
        np.sum(
            data == tokenizer.eos_id
        )
    )

    print(
        f"\n{source_name}"
    )

    print(
        "  tokens:",
        f"{token_count:,}",
    )

    print(
        "  min ID:",
        min_id,
    )

    print(
        "  max ID:",
        max_id,
    )

    print(
        "  EOS:",
        f"{eos_count:,}",
    )

    print(
        "  size:",
        f"{path.stat().st_size / (1024**3):.2f} GB",
    )

    assert min_id >= 0
    assert max_id < tokenizer.get_vocab_size()
    assert eos_count > 0

    del data


print("\n✓ All tokenized files verified")

TOKENIZED DATA VERIFICATION

acts
  tokens: 256,521,619
  min ID: 3
  max ID: 32003
  EOS: 15,490
  size: 0.48 GB

high_courts
  tokens: 242,356,017
  min ID: 3
  max ID: 32003
  EOS: 223,628
  size: 0.45 GB

supreme_court
  tokens: 241,501,699
  min ID: 3
  max ID: 32003
  EOS: 26,627
  size: 0.45 GB

tribunals
  tokens: 241,779,387
  min ID: 3
  max ID: 32003
  EOS: 57,691
  size: 0.45 GB

✓ All tokenized files verified


### CELL 14 · Final inventory

**In simple words:** Summary of the tokenized corpus.

In [15]:
# ============================================================
# CELL 14 — FINAL DATASET INVENTORY
# ============================================================

print("=" * 70)
print("TOKENIZED CORPUS")
print("=" * 70)

total_tokens = 0
total_bytes = 0

for path in sorted(
    OUTPUT_DIR.glob(
        "*.bin"
    )
):

    tokens = (
        path.stat().st_size
        // np.dtype(
            np.uint16
        ).itemsize
    )

    size_gb = (
        path.stat().st_size
        / (1024 ** 3)
    )

    total_tokens += tokens
    total_bytes += path.stat().st_size

    print(
        f"{path.name:30s}"
        f"{tokens:15,d} tokens"
        f"   {size_gb:.2f} GB"
    )

print("-" * 70)

print(
    f"{'TOTAL':30s}"
    f"{total_tokens:15,d} tokens"
    f"   {total_bytes / (1024**3):.2f} GB"
)

print(
    "\nMetadata:",
    metadata_path,
)

TOKENIZED CORPUS
acts_100m.bin                     256,521,619 tokens   0.48 GB
high_courts_100m.bin              242,356,017 tokens   0.45 GB
supreme_court_100m.bin            241,501,699 tokens   0.45 GB
tribunals_100m.bin                241,779,387 tokens   0.45 GB
----------------------------------------------------------------------
TOTAL                             982,158,722 tokens   1.83 GB

Metadata: /kaggle/working/legal_tokenized_100m/tokenization_metadata.json
